In [ ]:
# BATCH DIAMOND ANALYSIS FOR MULTIPLE EXCEL FILES
# This notebook processes multiple "Filtered-Erwinia amylovora IDXX.xlsx" files at once

# ============================================================================
# CELL 1: Install mambaforge
# ============================================================================
# @title Install mambaforge
import os
import shutil
import sys
import time

if 'google.colab' in sys.modules and not os.path.exists('/opt/conda/bin/mamba'):
    print('Installing mambaforge...')
    mambaforge_installer_url = 'https://github.com/conda-forge/miniforge/releases/download/24.3.0-0/Mambaforge-24.3.0-0-Linux-x86_64.sh'
    mambaforge_installer_path = 'mambaforge-installer.sh'
    !wget {mambaforge_installer_url} -O {mambaforge_installer_path}
    !bash {mambaforge_installer_path} -u -b -p /opt/conda
    !rm {mambaforge_installer_path}
    os.environ["PATH"] = "/opt/conda/bin:" + os.environ.get("PATH", "")
    print("Verifying mamba installation...")
    !mamba --version
    print("✅ mambaforge installation complete.")
else:
    print("mambaforge already installed or not in Colab environment.")

In [ ]:
# ============================================================================
# CELL 2: Install DIAMOND using mamba
# ============================================================================
# @title Install DIAMOND using mamba
import os

if "/opt/conda/bin" not in os.environ["PATH"]:
     os.environ["PATH"] = "/opt/conda/bin:" + os.environ.get("PATH", "")

print("Installing DIAMOND with mamba...")
!mamba install -c bioconda diamond -y

print("\nVerifying DIAMOND installation...")
!diamond --version
print("✅ DIAMOND installation successful.")

In [ ]:
# ============================================================================
# CELL 3: Download and Prepare the PHI-base Database (ONE TIME ONLY)
# ============================================================================
# @title Step 2: Download and Prepare the PHI-base Database
import os

if "/opt/conda/bin" not in os.environ["PATH"]:
     os.environ["PATH"] = "/opt/conda/bin:" + os.environ.get("PATH", "")

print("\nDownloading PHI-base database...")
phi_base_url = "https://raw.githubusercontent.com/PHI-base/data/master/releases/phi-base_v4-18_2025-05-28.fas"
phi_base_filename = "phi-base_v4-18_2025-05-28.fas"
!wget -q -O {phi_base_filename} {phi_base_url}
print("✅ Download complete.")

print("\nCreating DIAMOND database from PHI-base data...")
!diamond makedb --in {phi_base_filename} -d phi_base
print("✅ DIAMOND database 'phi_base.dmnd' created successfully.")

In [ ]:
# ============================================================================
# CELL 4: MOUNT GOOGLE DRIVE (RECOMMENDED FOR LARGE BATCH PROCESSING)
# ============================================================================
# @title Mount Google Drive (Optional but Recommended)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================================
# CELL 5: BATCH PROCESSING - MAIN ANALYSIS CODE
# ============================================================================
# @title BATCH PROCESSING: Analyze Multiple Excel Files

import pandas as pd
import os
import re
import numpy as np
from pathlib import Path
import glob

# --- CONFIGURATION ---
# You can modify these settings:

# Option 1: Upload files directly to Colab (for ~20-50 files)
# Just upload your Excel files to the current directory and they'll be auto-detected

# Option 2: Use files from Google Drive (for all 268 files)
# Set USE_GOOGLE_DRIVE = True and specify the path
USE_GOOGLE_DRIVE = False  # Change to True if using Google Drive
DRIVE_INPUT_PATH = "/content/drive/MyDrive/DIAMOND/Input"  # Your input folder path
DRIVE_OUTPUT_PATH = "/content/drive/MyDrive/DIAMOND/Output"  # Your output folder path

# File pattern to match
FILE_PATTERN = "Filtered-Erwinia amylovora ID*.xlsx"

# Sheet name (should be same for all files)
SHEET_NAME = 'Remaining_Proteins (2)'

# Column indices (0-based)
SEQUENCE_COLUMN_INDEX = 15  # Column P
ORGANISM_COLUMN_INDEX = 12   # Column M
FUNCTION_COLUMN_INDEX = 13   # Column N
DIAMOND_OUTPUT_COLUMN_INDEX = 16  # Column Q

# DIAMOND parameters
EVALUE_THRESHOLD = "1e-50"
MIN_IDENTITY = 50
REQUIRED_GAPS = 0

# --- END CONFIGURATION ---

print("=" * 80)
print("BATCH DIAMOND ANALYSIS - STARTING")
print("=" * 80)

# Determine input/output directories
if USE_GOOGLE_DRIVE:
    input_dir = DRIVE_INPUT_PATH
    output_dir = DRIVE_OUTPUT_PATH
    os.makedirs(output_dir, exist_ok=True)
else:
    input_dir = "/content"
    output_dir = "/content"

# Find all Excel files matching the pattern
excel_files = glob.glob(os.path.join(input_dir, FILE_PATTERN))
excel_files.sort()  # Sort by filename

if not excel_files:
    print(f"❌ ERROR: No files found matching pattern '{FILE_PATTERN}' in '{input_dir}'")
    print("\nPlease either:")
    print("  1. Upload Excel files to Colab (Files tab on left)")
    print("  2. Set USE_GOOGLE_DRIVE=True and specify correct paths")
else:
    print(f"\n✅ Found {len(excel_files)} files to process:")
    for i, f in enumerate(excel_files[:5], 1):  # Show first 5
        print(f"   {i}. {os.path.basename(f)}")
    if len(excel_files) > 5:
        print(f"   ... and {len(excel_files) - 5} more files")

# Function to extract ID from filename
def extract_id_from_filename(filename):
    """Extract ID number from filename like 'Filtered-Erwinia amylovora ID28.xlsx'"""
    match = re.search(r'ID(\d+)', filename)
    if match:
        return match.group(1)
    return None

# Function to parse PHI-base subject header
def parse_phi_header_corrected(header):
    parts = header.split('#')
    if len(parts) >= 2:
        phenotype = parts[-1].replace('_', ' ')
        organism = parts[-2].replace('_', ' ')
        return organism, phenotype
    return "Unknown Organism", "Unknown Function"

# --- MAIN PROCESSING LOOP ---
print("\n" + "=" * 80)
print("STARTING BATCH PROCESSING")
print("=" * 80 + "\n")

successful_files = []
failed_files = []

for file_idx, input_excel_file in enumerate(excel_files, 1):
    try:
        # Extract ID from filename
        file_id = extract_id_from_filename(os.path.basename(input_excel_file))
        if not file_id:
            print(f"⚠️  Skipping {os.path.basename(input_excel_file)} - Could not extract ID")
            failed_files.append((input_excel_file, "Could not extract ID"))
            continue

        print(f"\n{'='*80}")
        print(f"Processing [{file_idx}/{len(excel_files)}]: ID{file_id}")
        print(f"{'='*80}")
        print(f"Input file: {os.path.basename(input_excel_file)}")

        # Create output filenames (no parentheses to avoid shell issues)
        fasta_query_file = f"query_sequences_ID{file_id}.fasta"
        diamond_output_file = f"diamond_results_ID{file_id}.tsv"
        process_log_file = f"Process_and_Filter_Results_ID{file_id}.txt"
        final_excel_file = f"FINAL_ANNOTATED_Filtered-Erwinia_amylovora_ID{file_id}.xlsx"

        # Redirect print output to log file
        import sys
        from io import StringIO
        log_buffer = StringIO()

        # Step 1: Read Excel file
        print(f"📖 Reading Excel sheet: {SHEET_NAME}...")
        df_original = pd.read_excel(input_excel_file, sheet_name=SHEET_NAME, header=None)
        log_msg = f"✅ Successfully loaded {len(df_original)} rows from the sheet."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Step 2: Convert sequences to FASTA
        print("🧬 Converting sequences to FASTA format...")
        fasta_string = ""
        for index, row in df_original.iterrows():
            sequence = str(row[SEQUENCE_COLUMN_INDEX])
            header = f">seq{index + 1}"
            fasta_string += f"{header}\n{sequence}\n"

        with open(fasta_query_file, "w") as f:
            f.write(fasta_string)
        log_msg = "✅ Converted sequences to FASTA format."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Step 3: Run DIAMOND alignment
        print("⚡ Running DIAMOND alignment...")
        diamond_cmd = f"diamond blastp -d phi_base -q '{fasta_query_file}' -o '{diamond_output_file}' -e {EVALUE_THRESHOLD} --outfmt 6 qseqid sseqid pident gaps --threads {os.cpu_count()} --quiet"
        os.system(diamond_cmd)
        log_msg = "✅ DIAMOND alignment complete."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Step 4: Process and filter results
        print("🔍 Processing and filtering results...")
        col_names = ['query_id', 'subject_id', 'identity', 'gaps']
        df_results = pd.read_csv(diamond_output_file, sep='\t', header=None, names=col_names)
        log_msg = f"Found {len(df_results)} total alignments."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Apply filters
        df_filtered = df_results[
            (df_results['gaps'] == REQUIRED_GAPS) &
            (df_results['identity'] >= MIN_IDENTITY)
        ].copy()
        log_msg = f"Found {len(df_filtered)} alignments meeting the criteria (gaps={REQUIRED_GAPS}, identity>={MIN_IDENTITY})."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Process results
        results_dict = {}

        for query_id, group in df_filtered.groupby('query_id'):
            hits_100 = group[group['identity'] == 100.0]

            if not hits_100.empty:
                organisms = set()
                phenotypes = set()
                raw_outputs = []

                for _, row in hits_100.iterrows():
                    organism, phenotype = parse_phi_header_corrected(row['subject_id'])
                    organisms.add(organism)
                    phenotypes.add(phenotype)
                    raw_outputs.append(f"{row['subject_id']}\t{row['identity']}\t{row['gaps']}")

                results_dict[query_id] = {
                    'identity': 100.0,
                    'organism': ', '.join(sorted(list(organisms))),
                    'function': ', '.join(sorted(list(phenotypes))),
                    'raw_output': '\n'.join(raw_outputs)
                }
            else:
                best_hit = group.loc[group['identity'].idxmax()]
                organism, phenotype = parse_phi_header_corrected(best_hit['subject_id'])
                raw_output_string = f"{best_hit['subject_id']}\t{best_hit['identity']}\t{best_hit['gaps']}"

                results_dict[query_id] = {
                    'identity': best_hit['identity'],
                    'organism': organism,
                    'function': phenotype,
                    'raw_output': raw_output_string
                }

        log_msg = f"✅ Processed and consolidated results for {len(results_dict)} unique proteins."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Step 5: Annotate DataFrame
        print("📝 Annotating original DataFrame...")
        for col_idx in [ORGANISM_COLUMN_INDEX, FUNCTION_COLUMN_INDEX, DIAMOND_OUTPUT_COLUMN_INDEX]:
            if col_idx not in df_original.columns:
                df_original[col_idx] = np.nan
            df_original[col_idx] = df_original[col_idx].astype(object)

        for seq_id, data in results_dict.items():
            row_index = int(seq_id.replace('seq', '')) - 1
            if row_index < len(df_original):
                df_original.at[row_index, ORGANISM_COLUMN_INDEX] = data['organism']
                df_original.at[row_index, FUNCTION_COLUMN_INDEX] = data['function']
                df_original.at[row_index, DIAMOND_OUTPUT_COLUMN_INDEX] = data['raw_output']

        log_msg = "✅ Annotation columns populated."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Step 6: Apply styling and save
        print("🎨 Applying row highlighting...")
        def highlight_rows(row):
            seq_id = f"seq{row.name + 1}"
            if seq_id in results_dict:
                identity = results_dict[seq_id]['identity']
                if identity == 100.0:
                    return ['background-color: lightgreen'] * len(row)
                elif identity >= MIN_IDENTITY:
                    return ['background-color: yellow'] * len(row)
            return [''] * len(row)

        styled_df = df_original.style.apply(highlight_rows, axis=1)
        log_msg = "✅ Row highlighting rules applied."
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Save final Excel file
        output_excel_path = os.path.join(output_dir, final_excel_file)
        styled_df.to_excel(output_excel_path, index=False, header=False, engine='openpyxl')
        log_msg = f"💾 Saved: {final_excel_file}"
        print(log_msg)
        log_buffer.write(log_msg + "\n")

        # Save process log
        log_path = os.path.join(output_dir, process_log_file)
        with open(log_path, 'w') as log_file:
            log_file.write(f"Process and Filter Results for ID{file_id}\n")
            log_file.write("=" * 80 + "\n\n")
            log_file.write(log_buffer.getvalue())
        print(f"📄 Saved: {process_log_file}")

        # Move intermediate files to output directory if using Google Drive
        if USE_GOOGLE_DRIVE:
            import shutil
            shutil.move(fasta_query_file, os.path.join(output_dir, fasta_query_file))
            shutil.move(diamond_output_file, os.path.join(output_dir, diamond_output_file))

        print(f"\n✅ SUCCESS: ID{file_id} processing complete!")
        successful_files.append(input_excel_file)

    except Exception as e:
        error_msg = f"❌ ERROR processing ID{file_id}: {str(e)}"
        print(error_msg)
        failed_files.append((input_excel_file, str(e)))

        # Save error log
        try:
            error_log_path = os.path.join(output_dir, f"ERROR_ID{file_id}.txt")
            with open(error_log_path, 'w') as error_file:
                error_file.write(f"Error processing ID{file_id}\n")
                error_file.write("=" * 80 + "\n\n")
                error_file.write(str(e) + "\n\n")
                import traceback
                error_file.write(traceback.format_exc())
        except:
            pass

# --- FINAL SUMMARY ---
print("\n" + "=" * 80)
print("BATCH PROCESSING COMPLETE - SUMMARY")
print("=" * 80)
print(f"\n✅ Successfully processed: {len(successful_files)} files")
print(f"❌ Failed: {len(failed_files)} files")

if successful_files:
    print(f"\n📊 Successfully processed files:")
    for f in successful_files[:10]:
        print(f"   ✓ {os.path.basename(f)}")
    if len(successful_files) > 10:
        print(f"   ... and {len(successful_files) - 10} more")

if failed_files:
    print(f"\n⚠️  Failed files:")
    for f, error in failed_files:
        print(f"   ✗ {os.path.basename(f)}: {error}")

print(f"\n📁 Output location: {output_dir}")
print("\n" + "=" * 80)

In [ ]:
# ============================================================================
# CELL 6: DOWNLOAD ALL OUTPUT FILES (Optional - for Colab storage)
# ============================================================================
# @title Download All Output Files as ZIP

import zipfile
import os

if not USE_GOOGLE_DRIVE:
    print("Creating ZIP file with all outputs...")

    zip_filename = "DIAMOND_Batch_Results.zip"

    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add all output files
        for file in glob.glob("FINAL_ANNOTATED_*.xlsx"):
            zipf.write(file)
        for file in glob.glob("diamond_results_ID*.tsv"):
            zipf.write(file)
        for file in glob.glob("query_sequences_ID*.fasta"):
            zipf.write(file)
        for file in glob.glob("Process_and_Filter*.txt"):
            zipf.write(file)

    print(f"✅ Created {zip_filename}")
    print("\nDownload the ZIP file from the Files panel (left sidebar)")

    # Trigger download
    from google.colab import files
    files.download(zip_filename)
else:
    print("Files are already saved to Google Drive!")
    print(f"Location: {output_dir}")

In [ ]:
# ============================================================================
# CELL 7: CLEANUP - Remove All Processing Files from Session
# ============================================================================
# @title CLEANUP: Remove All Input/Output Files from Session Storage

import os
import glob
import shutil

print("=" * 80)
print("CLEANUP: REMOVING FILES FROM SESSION STORAGE")
print("=" * 80)

# Only clean up if NOT using Google Drive (files already saved there)
if not USE_GOOGLE_DRIVE:
    print("\n⚠️  WARNING: This will delete all input and output files from Colab session!")
    print("Make sure you have downloaded the ZIP file first!")

    # You can uncomment the line below to require manual confirmation
    # confirmation = input("\nType 'DELETE' to proceed: ")
    # if confirmation != "DELETE":
    #     print("❌ Cleanup cancelled.")
    #     exit()

# List of file patterns to remove
file_patterns_to_remove = [
    "Filtered-Erwinia amylovora ID*.xlsx",  # Input Excel files
    "FINAL_ANNOTATED_*.xlsx",                # Output Excel files
    "query_sequences_ID*.fasta",             # FASTA query files
    "diamond_results_ID*.tsv",               # DIAMOND result files
    "Process_and_Filter_Results_ID*.txt",    # Process log files
    "ERROR_ID*.txt",                         # Error log files
    "DIAMOND_Batch_Results.zip"              # ZIP file
]

files_removed = 0
total_size_freed = 0

print("\n🗑️  Removing files...\n")

for pattern in file_patterns_to_remove:
    matching_files = glob.glob(pattern)
    for file_path in matching_files:
        try:
            file_size = os.path.getsize(file_path)
            os.remove(file_path)
            files_removed += 1
            total_size_freed += file_size
            print(f"   ✓ Removed: {os.path.basename(file_path)} ({file_size / 1024:.1f} KB)")
        except Exception as e:
            print(f"   ✗ Failed to remove {os.path.basename(file_path)}: {e}")

print(f"\n{'='*80}")
print(f"✅ CLEANUP COMPLETE")
print(f"{'='*80}")
print(f"   Files removed: {files_removed}")
print(f"   Space freed: {total_size_freed / (1024*1024):.2f} MB")
print(f"\n📂 Session storage is now clean and ready for the next batch!")
print(f"{'='*80}\n")

# Optional: Show remaining files in current directory
remaining_files = [f for f in os.listdir('.') if os.path.isfile(f)]
print(f"\n📋 Remaining files in session ({len(remaining_files)} files):")
if remaining_files:
    # Show only non-system files
    user_files = [f for f in remaining_files if not f.startswith('.') and f not in ['sample_data']]
    if user_files:
        for f in user_files[:20]:  # Show first 20
            print(f"   • {f}")
        if len(user_files) > 20:
            print(f"   ... and {len(user_files) - 20} more files")
    else:
        print("   ✅ No user files remaining (only system files)")
else:
    print("   ✅ Directory is clean!")

print("\n💡 TIP: You can now upload the next batch of Excel files (20-50 files)")
print("   and run Cell 5 again to process them.")